In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

current_dir = Path.cwd().resolve()

if current_dir.name == "notebooks":
    CV_MODEL = current_dir.parent
elif current_dir.name == "cv_model":
    CV_MODEL = current_dir
else:
    CV_MODEL = current_dir / "cv_model"

# Исходная модель
MODEL_PATH = CV_MODEL / "models" / "pretrained" / "yolo11n.pt"

# Итоговый датасет
DATA_PATH = CV_MODEL / "yolo_person_dataset" / "data.yaml"

print("Модель найдена:", MODEL_PATH.exists())
print("Датасет найден:", DATA_PATH.exists())

Модель найдена: True
Датасет найден: True


In [11]:
# Загружаем исходную YOLO11n.
# Это baseline-модель: она ещё не обучалась на наших изображениях.
model = YOLO(MODEL_PATH)


# Один раз проверяем модель на вручную размеченном test-наборе.
# Функция val() только оценивает модель и не изменяет её веса.
metrics = model.val(
    data=str(DATA_PATH),  # путь к описанию датасета
    split="test",         # используем тестовую часть
    classes=[0],          # оцениваем только класс person
    imgsz=640,            # размер изображения для YOLO
    device="mps",         # используем графический процессор Mac M3
    plots=False,          # не строим огромную матрицу по 80 классам COCO
)

Ultralytics 8.4.115 🚀 Python-3.13.3 torch-2.12.1 MPS (Apple M3)
YOLO11n summary (fused): 100 layers, 2,616,248 parameters, 0 gradients, 6.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2482.7±523.3 MB/s, size: 66.0 KB)
val: Scanning /Users/kristinaananova/Desktop/SummerPractice2026/cv_model/yolo_test_dataset/labels/test/camera_1.cache... 97 images, 23 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 120/120 62.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 5.4it/s 1.5s0.2s
                   all        120         97       0.98      0.498      0.829      0.609
                person         97         97       0.98      0.498      0.829      0.609
Speed: 0.1ms preprocess, 4.5ms inference, 0.0ms loss, 2.9ms postprocess per image


In [12]:
# Забираем уже рассчитанные метрики из результата model.val().
baseline_metrics = pd.DataFrame(
    {
        "Метрика": [
            "Precision",
            "Recall",
            "mAP50",
            "mAP50-95",
        ],
        "Значение": [
            metrics.box.mp,
            metrics.box.mr,
            metrics.box.map50,
            metrics.box.map,
        ],
    }
)

# Округляем значения для удобного отображения.
baseline_metrics["Значение"] = (
    baseline_metrics["Значение"].round(4)
)

# Показываем таблицу.
display(baseline_metrics)


# Строим простой график по тем же готовым метрикам.
ax = baseline_metrics.plot.bar(
    x="Метрика",
    y="Значение",
    legend=False,
    figsize=(8, 5),
    color="#4C78A8",
)

# Все метрики находятся в диапазоне от 0 до 1.
ax.set_ylim(0, 1)

ax.set_title("Качество исходной YOLO11n на test")
ax.set_xlabel("")
ax.set_ylabel("Значение метрики")
ax.tick_params(axis="x", rotation=0)

# Подписываем точные значения над столбцами.
for container in ax.containers:
    ax.bar_label(container, fmt="%.4f")

plt.tight_layout()
plt.show()

,Метрика,Значение
0,Precision,0.9797
1,Recall,0.4983
2,mAP50,0.8291
3,mAP50-95,0.6091


<Figure size 800x500 with 1 Axes>